# Atelier Préparation de Données Tabulaires

**Contexte** : préparation d'un jeu de données de capteurs IoT issus de bâtiments intelligents
(température, humidité, CO₂, consommation énergétique, occupation...) en vue d'un modèle de
Machine Learning (prédiction de consommation / détection d'anomalies).

Ce notebook suit la structure de l'atelier, **question par question**.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")
%matplotlib inline

---
# Partie 1 – Explorer les données

### 1) Charger les données CSV

In [3]:
df = pd.read_csv("../data/smart_building_raw.csv")
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


### 2) Afficher les premières lignes du dataset

In [4]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


### 3) Afficher les dernières lignes du dataset

In [5]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


### 4) Combien d'observations contient le dataset ?

In [6]:
n_obs = df.shape[0]
print(f"Le dataset contient {n_obs} observations (lignes).")

Le dataset contient 507 observations (lignes).


### 5) Combien de variables possède le dataset ?

In [7]:
n_var = df.shape[1]
print(f"Le dataset possède {n_var} variables (colonnes).")
df.columns.tolist()

Le dataset possède 14 variables (colonnes).


['id_mesure',
 'date',
 'batiment',
 'type_batiment',
 'zone',
 'temperature',
 'humidite',
 'co2',
 'occupation',
 'consommation_kwh',
 'mode_climatisation',
 'etat_systeme',
 'jour_semaine',
 'alerte']

### 6) Identifier les variables numériques

On utilise `select_dtypes` pour repérer les colonnes de type numérique.
`id_mesure` est numérique au sens du type mais c'est en réalité un **identifiant**
(cf. question 9), on l'exclut donc des variables numériques "utiles".

In [8]:
variables_numeriques = df.select_dtypes(include=[np.number]).columns.tolist()
print("Variables numériques (type) :", variables_numeriques)

variables_numeriques_utiles = [c for c in variables_numeriques if c != "id_mesure"]
print("Variables numériques utiles (hors identifiant) :", variables_numeriques_utiles)

Variables numériques (type) : ['id_mesure', 'temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']
Variables numériques utiles (hors identifiant) : ['temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']


### 7) Identifier les variables catégorielles

Colonnes de type `object` (texte), hors la date qui est une variable temporelle à part.

In [9]:
variables_categorielles = df.select_dtypes(include="object").columns.tolist()
variables_categorielles = [c for c in variables_categorielles if c != "date"]
print("Variables catégorielles :", variables_categorielles)

Variables catégorielles : ['batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


/var/folders/0h/_d5jczk96_v7v7jl988tw0xm0000gn/T/ipykernel_91315/4155203364.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  variables_categorielles = df.select_dtypes(include="object").columns.tolist()


### 8) Identifier les dates

On tente la conversion en `datetime` avec `errors="coerce"` : les dates mal formées ou
impossibles (ex : `2025-02-30`, `2025-13-45`, `"date_invalide"`) sont alors transformées en
`NaT` plutôt que de faire planter le chargement — un problème de qualité supplémentaire à
traiter comme les autres valeurs manquantes.

In [10]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
variables_dates = ["date"]
print("Variable(s) de type date :", variables_dates)

nb_dates_invalides = df["date"].isna().sum()
print(f"{nb_dates_invalides} dates invalides transformées en NaT (valeurs manquantes).")
df["date"].head()

Variable(s) de type date : ['date']
5 dates invalides transformées en NaT (valeurs manquantes).


0   2025-02-13 06:00:00
1   2025-03-10 12:00:00
2   2025-05-04 00:00:00
3   2025-01-19 00:00:00
4   2025-04-24 06:00:00
Name: date, dtype: datetime64[us]

### 9) Identifier les identifiants

`id_mesure` est un identifiant unique de chaque mesure (une ligne = un identifiant).
Il n'a aucune valeur prédictive et doit être exclu des variables explicatives du modèle.

In [11]:
print("id_mesure est-il unique sur toutes les lignes ?", df["id_mesure"].is_unique)
variables_identifiants = ["id_mesure"]
print("Variable(s) identifiant(s) :", variables_identifiants)

id_mesure est-il unique sur toutes les lignes ? False
Variable(s) identifiant(s) : ['id_mesure']


### 10) Statistiques descriptives : moyenne, médiane, min, max, écart-type, quartiles

In [12]:
df[variables_numeriques_utiles].describe().T.assign(
    mediane=df[variables_numeriques_utiles].median()
)[["mean","mediane","std","min","25%","50%","75%","max"]]

,mean,mediane,std,min,25%,50%,75%,max
temperature,24.154141,24.00,7.418465,-30.0,21.600,24.00,26.000,96.0
humidite,57.864113,57.55,16.026336,-12.0,49.275,57.55,65.750,160.0
co2,844.150000,787.50,582.181386,89.0,623.750,787.50,952.000,6000.0
occupation,44.850299,46.00,24.949139,-20.0,27.000,46.00,61.000,116.0
consommation_kwh,169.069323,169.80,53.164294,-100.0,136.875,169.80,202.975,336.2


### 11) Y a-t-il des variables potentiellement problématiques ?

D'après les statistiques ci-dessus, plusieurs variables présentent des valeurs suspectes :

- **temperature** : minimum très bas et maximum très élevé (valeurs physiquement improbables
  pour un bâtiment) ;
- **humidite** : devrait être bornée entre 0 et 100 (%), or on observe des valeurs hors de cet
  intervalle ;
- **co2** : présence de valeurs extrêmement élevées comparées au reste de la distribution ;
- **occupation** : ne peut pas être négative (nombre de personnes) ;
- **consommation_kwh** : ne peut pas être négative ;
- **type_batiment**, **mode_climatisation** : variables catégorielles avec probablement des
  catégories mal orthographiées / mal normalisées (espaces, casse).

On explore ces points en détail dans la suite.

### 12) Données incohérentes

**a) Humidité < 0**

In [13]:
mask_humidite_neg = df["humidite"] < 0
print(f"{mask_humidite_neg.sum()} lignes avec une humidité négative.")
df.loc[mask_humidite_neg, ["id_mesure", "humidite"]]

3 lignes avec une humidité négative.


,id_mesure,humidite
281,1126,-5.0
335,1036,-8.0
366,1216,-12.0


**b) Humidité > 100**

In [14]:
mask_humidite_sup100 = df["humidite"] > 100
print(f"{mask_humidite_sup100.sum()} lignes avec une humidité > 100%.")
df.loc[mask_humidite_sup100, ["id_mesure", "humidite"]]

7 lignes avec une humidité > 100%.


,id_mesure,humidite
59,1246,108.0
103,1016,145.0
128,1156,160.0
160,1186,125.0
268,1276,140.0
327,1066,132.0
342,1096,110.0


**c) Température extrêmement élevée (ou basse) — on fixe un seuil physiquement raisonnable pour un bâtiment (ex : entre -10°C et 45°C)**

In [15]:
mask_temp_aberrante = (df["temperature"] < -10) | (df["temperature"] > 45)
print(f"{mask_temp_aberrante.sum()} lignes avec une température hors de [-10°C, 45°C].")
df.loc[mask_temp_aberrante, ["id_mesure", "temperature"]]

6 lignes avec une température hors de [-10°C, 45°C].


,id_mesure,temperature
7,1141,72.5
116,1181,96.0
161,1061,88.0
185,1221,-25.0
358,1101,-30.0
499,1021,95.2


**d) Occupation négative**

In [16]:
mask_occupation_neg = df["occupation"] < 0
print(f"{mask_occupation_neg.sum()} lignes avec une occupation négative.")
df.loc[mask_occupation_neg, ["id_mesure", "occupation"]]

5 lignes avec une occupation négative.


,id_mesure,occupation
22,1031,-5.0
376,1231,-8.0
430,1081,-12.0
487,1131,-2.0
494,1331,-20.0


**e) Consommation négative**

In [17]:
mask_conso_neg = df["consommation_kwh"] < 0
print(f"{mask_conso_neg.sum()} lignes avec une consommation négative.")
df.loc[mask_conso_neg, ["id_mesure", "consommation_kwh"]]

4 lignes avec une consommation négative.


,id_mesure,consommation_kwh
59,1246,-15.0
154,1046,-50.0
199,1146,-20.0
441,1346,-100.0


In [18]:
df_clean = df.copy()

df_clean.loc[mask_humidite_neg | mask_humidite_sup100, "humidite"] = np.nan
df_clean.loc[mask_temp_aberrante, "temperature"] = np.nan
df_clean.loc[mask_occupation_neg, "occupation"] = np.nan
df_clean.loc[mask_conso_neg, "consommation_kwh"] = np.nan

print("Nombre de valeurs transformées en NaN par cette étape :")
print("humidite     :", (mask_humidite_neg | mask_humidite_sup100).sum())
print("temperature  :", mask_temp_aberrante.sum())
print("occupation   :", mask_occupation_neg.sum())
print("consommation :", mask_conso_neg.sum())

Nombre de valeurs transformées en NaN par cette étape :
humidite     : 10
temperature  : 6
occupation   : 5
consommation : 4


**g) Catégories mal orthographiées**

On regarde les valeurs uniques des variables catégorielles : on y trouve des variantes
d'écriture d'une même catégorie (espaces superflus, majuscules/minuscules différentes,
fautes de frappe comme `"Bureu"` pour `"Bureau"`, `"entrepot"` sans accent pour `"Entrepôt"`,
`"normale"` pour `"Normal"`, etc.).

In [19]:
for c in ["type_batiment", "mode_climatisation", "etat_systeme"]:
    print(f"--- {c} ---")
    print(sorted(df_clean[c].dropna().unique().tolist()))
    print()

--- type_batiment ---
[' Bureau ', ' UNIVERSITÉ', 'BUREAU', 'Bureau', 'Bureu', 'Centre commercial', 'Entrepôt', 'Hôpital', 'Université', 'bureau', 'centre commercial', 'ecole', 'entrepot', 'hôpital ', 'ÉCOLE', 'École']

--- mode_climatisation ---
['BOOST', 'Boost', 'Eco', 'Normal', 'Normal ', 'normal', 'normale']

--- etat_systeme ---
['Alerte', 'Normal', 'Panne']



**h) Normaliser les catégories textuelles : suppression des espaces puis uniformisation de la casse**

In [20]:
for c in ["type_batiment", "mode_climatisation", "etat_systeme", "jour_semaine"]:
    df_clean[c] = df_clean[c].str.strip().str.lower()

# Correction des fautes de frappe / variantes restantes après normalisation de la casse
mapping_type_batiment = {
    "bureau": "bureau",
    "bureu": "bureau",          # faute de frappe
    "université": "universite",
    "universite": "universite",
    "entrepôt": "entrepot",
    "entrepot": "entrepot",
    "hôpital": "hopital",
    "hopital": "hopital",
    "école": "ecole",
    "ecole": "ecole",
    "centre commercial": "centre commercial",
}
mapping_climatisation = {
    "eco": "eco",
    "normal": "normal",
    "normale": "normal",        # faute de frappe
    "boost": "boost",
}

df_clean["type_batiment"] = df_clean["type_batiment"].replace(mapping_type_batiment)
df_clean["mode_climatisation"] = df_clean["mode_climatisation"].replace(mapping_climatisation)

print("type_batiment :", sorted(df_clean["type_batiment"].dropna().unique().tolist()))
print("mode_climatisation :", sorted(df_clean["mode_climatisation"].dropna().unique().tolist()))
print("etat_systeme :", sorted(df_clean["etat_systeme"].dropna().unique().tolist()))
print("jour_semaine :", sorted(df_clean["jour_semaine"].dropna().unique().tolist()))

type_batiment : ['bureau', 'centre commercial', 'ecole', 'entrepot', 'hopital', 'universite']
mode_climatisation : ['boost', 'eco', 'normal']
etat_systeme : ['alerte', 'normal', 'panne']
jour_semaine : ['dimanche', 'jeudi', 'lundi', 'mardi', 'mercredi', 'samedi', 'vendredi']


### 13) Valeurs manquantes

**a) Nombre et pourcentage de valeurs manquantes par colonne**

In [21]:
missing_count = df_clean.isna().sum()
missing_pct = (df_clean.isna().mean() * 100).round(2)
missing_summary = pd.DataFrame({
    "nb_manquantes": missing_count,
    "pct_manquantes": missing_pct
}).sort_values("nb_manquantes", ascending=False)
missing_summary

,nb_manquantes,pct_manquantes
humidite,21,4.14
temperature,18,3.55
occupation,11,2.17
consommation_kwh,9,1.78
co2,7,1.38
date,5,0.99
mode_climatisation,5,0.99
jour_semaine,5,0.99
type_batiment,4,0.79
id_mesure,0,0.00


**b) Quelle variable possède le plus de valeurs manquantes ?**

In [22]:
var_plus_manquante = missing_summary.index[0]
print(f"La variable avec le plus de valeurs manquantes est : '{var_plus_manquante}' "
      f"({missing_summary.iloc[0]['nb_manquantes']:.0f} valeurs, "
      f"{missing_summary.iloc[0]['pct_manquantes']}%).")

La variable avec le plus de valeurs manquantes est : 'humidite' (21 valeurs, 4.14%).


**c) Quelle stratégie utiliser pour les valeurs manquantes ?**

Le taux de valeurs manquantes est faible pour toutes les colonnes concernées (moins de ~5%),
ce qui permet une **imputation** plutôt qu'une suppression massive de lignes :

- variables numériques (temperature, humidite, co2, occupation, consommation_kwh) →
  imputation par la **médiane**, plus robuste aux valeurs extrêmes restantes que la moyenne ;
- variables catégorielles (type_batiment, mode_climatisation, jour_semaine) →
  imputation par le **mode** (valeur la plus fréquente).

C'est exactement la stratégie qui sera formalisée dans le pipeline scikit-learn (Partie 6).

**d) Peut-on supprimer toutes les lignes contenant des valeurs manquantes ?**

Techniquement oui, mais ce n'est pas recommandé ici : cumulées sur toutes les colonnes, les
lignes concernées par au moins une valeur manquante représentent une part non négligeable du
dataset (voir calcul ci-dessous). Supprimer ces lignes ferait perdre de l'information utile et
pourrait biaiser le jeu de données (ex : si les valeurs manquantes ne sont pas réparties au
hasard entre bâtiments ou entre classes de la cible `alerte`).

**e) Dans quels cas utiliser la moyenne ?**
Quand la distribution de la variable est à peu près symétrique et sans valeurs extrêmes
marquées : la moyenne résume alors bien la tendance centrale.

**f) Quand préférer la médiane ?**
Quand la distribution est asymétrique ou contient des valeurs extrêmes (outliers) : la médiane
n'est pas influencée par ces valeurs, contrairement à la moyenne. C'est le cas de nos variables
(co2, consommation_kwh...) qui contiennent des valeurs extrêmes.

**g) Comment traiter une variable catégorielle ?**
On ne peut pas calculer de moyenne/médiane sur du texte : on impute généralement par le
**mode** (catégorie la plus fréquente), ou on crée une catégorie explicite `"Inconnu"` si
l'absence de valeur est elle-même une information potentiellement utile.

In [23]:
lignes_avec_manquant = df_clean.isna().any(axis=1).sum()
print(f"{lignes_avec_manquant} lignes sur {len(df_clean)} "
      f"({lignes_avec_manquant/len(df_clean)*100:.1f}%) contiennent au moins une valeur manquante.")

73 lignes sur 507 (14.4%) contiennent au moins une valeur manquante.


### 14) Doublons

**a) Identifier les doublons**

In [24]:
doublons_mask = df_clean.duplicated(keep=False)
print(f"{df_clean.duplicated().sum()} lignes sont des doublons (en comptant une occurrence par groupe).")
print(f"{doublons_mask.sum()} lignes au total appartiennent à un groupe de doublons.")

7 lignes sont des doublons (en comptant une occurrence par groupe).
14 lignes au total appartiennent à un groupe de doublons.


**b) Afficher les doublons**

In [25]:
df_clean[doublons_mask].sort_values("id_mesure")

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
192,1026,2025-01-07 06:00:00,B6,bureau,A,23.0,74.3,952.0,89.0,247.6,normal,normal,mardi,Oui
117,1026,2025-01-07 06:00:00,B6,bureau,A,23.0,74.3,952.0,89.0,247.6,normal,normal,mardi,Oui
9,1118,2025-01-30 06:00:00,B1,bureau,A,22.0,NaN,479.0,64.0,145.0,normal,normal,jeudi,Non
123,1118,2025-01-30 06:00:00,B1,bureau,A,22.0,NaN,479.0,64.0,145.0,normal,normal,jeudi,Non
23,1264,2025-03-07 18:00:00,B1,bureau,B,21.1,38.3,1034.0,74.0,199.2,normal,panne,vendredi,Oui
434,1264,2025-03-07 18:00:00,B1,bureau,B,21.1,38.3,1034.0,74.0,199.2,normal,panne,vendredi,Oui
455,1320,2025-03-21 18:00:00,B7,bureau,B,27.5,27.2,359.0,33.0,185.8,normal,normal,vendredi,Non
464,1320,2025-03-21 18:00:00,B7,bureau,B,27.5,27.2,359.0,33.0,185.8,normal,normal,vendredi,Non
238,1402,2025-04-11 06:00:00,B2,ecole,A,22.6,44.0,868.0,62.0,172.5,eco,normal,vendredi,Non
418,1402,2025-04-11 06:00:00,B2,ecole,A,22.6,44.0,868.0,62.0,172.5,eco,normal,vendredi,Non


**c) Sont-ils réellement identiques ?**

On compare les doublons en excluant `id_mesure` (qui est par construction unique) pour voir si
le *contenu* des mesures est strictement identique.

In [26]:
cols_hors_id = [c for c in df_clean.columns if c != "id_mesure"]
doublons_contenu = df_clean.duplicated(subset=cols_hors_id, keep=False)
print(f"{doublons_contenu.sum()} lignes ont un contenu strictement identique (hors id_mesure).")
# Comparaison : sont-ils identiques uniquement sur id_mesure (faux doublons) ou sur tout le reste ?
print("Les doublons détectés en (a) le sont-ils aussi hors id_mesure ?",
      (doublons_mask == doublons_contenu).all())

14 lignes ont un contenu strictement identique (hors id_mesure).
Les doublons détectés en (a) le sont-ils aussi hors id_mesure ? True
